In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn tensorflow numpy

In [ ]:
import pandas as pd

colonnes_nutriscore = [
    "energy-kj_100g",
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "unsaturated-fat_100g",
    "omega-3-fat_100g",
    "sugars_100g",
    "added-sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "cholesterol_100g",
    "vitamin-a_100g",
    "vitamin-c_100g",
    "vitamin-d_100g",
    "calcium_100g",
    "magnesium_100g",
    "zinc_100g",
    "manganese_100g",
    "caffeine_100g",
    "alcohol_100g",
    "nutriscore_grade",
    "product_name"
]

# Lire uniquement les colonnes disponibles dans le fichier
df_header = pd.read_csv("../datas/FoodProduct_sample.csv", nrows=0)
colonnes_disponibles = df_header.columns.tolist()

# Filtrer la liste pour ne garder que les colonnes existantes
colonnes_a_charger = [col for col in colonnes_nutriscore if col in colonnes_disponibles]

print(f"Colonnes demand?es : {len(colonnes_nutriscore)}")
print(f"Colonnes trouv?es dans le fichier : {len(colonnes_disponibles)}")
print(f"Colonnes qui seront charg?es : {len(colonnes_a_charger)}")
print(f"Colonnes manquantes (non trouv?es) : {[col for col in colonnes_nutriscore if col not in colonnes_disponibles]}")

# Charger uniquement les colonnes existantes
df = pd.read_csv("../datas/FoodProduct_sample.csv",
    encoding='utf-8',
    usecols=colonnes_a_charger
)
# Garder uniquement colonnes existantes
colonnes_extraites = [col for col in colonnes_nutriscore if col in df.columns]
df_nutriscore = df[colonnes_extraites]

# Supprimer lignes avec nutriscore_grade non valide
valeurs_a_supprimer = ["unknown", "unknow", "not-applicable"]
df_nutriscore = df_nutriscore[~df_nutriscore["nutriscore_grade"].isin(valeurs_a_supprimer)]

# Calculer la moyenne ?nergie par Nutri-Score (exclure NaN)
moyenne_energie_par_score_kj = df_nutriscore.groupby("nutriscore_grade")["energy-kj_100g"].mean()
moyenne_energie_par_score_kcal = df_nutriscore.groupby("nutriscore_grade")["energy-kcal_100g"].mean()

# Fonction g?n?rique pour remplacer NaN ?nergie par moyenne selon Nutri-Score
def remplacer_energie(row, colonne, moyenne_par_score):
    if pd.isna(row[colonne]):
        return moyenne_par_score.get(row["nutriscore_grade"], 0)
    return row[colonne]

# Appliquer remplacement pour energy-kj_100g
df_nutriscore["energy-kj_100g"] = df_nutriscore.apply(lambda row: remplacer_energie(row, "energy-kj_100g", moyenne_energie_par_score_kj), axis=1)

# Appliquer remplacement pour energy-kcal_100g
df_nutriscore["energy-kcal_100g"] = df_nutriscore.apply(lambda row: remplacer_energie(row, "energy-kcal_100g", moyenne_energie_par_score_kcal), axis=1)

# Pour les autres colonnes num?riques, remplacer NaN par 0
colonnes_num = df_nutriscore.columns.difference(["nutriscore_grade", "energy-kj_100g", "energy-kcal_100g"])
df_nutriscore[colonnes_num] = df_nutriscore[colonnes_num].fillna(0)


# Renommer colonnes pour plus de clart? (exemple, adapte si tu veux)
renommage_colonnes = {
    "energy-kj_100g": "?nergie (kJ/100g)",
    "energy-kcal_100g": "?nergie (kcal/100g)",
    "fat_100g": "Mati?res grasses (g/100g)",
    "saturated-fat_100g": "Graisses satur?es (g/100g)",
    "unsaturated-fat_100g": "Graisses insatur?es (g/100g)",
    "omega-3-fat_100g": "Om?ga 3 (g/100g)",
    "sugars_100g": "Sucres (g/100g)",
    "added-sugars_100g": "Sucres ajout?s (g/100g)",
    "fiber_100g": "Fibres totales (g/100g)",
    "proteins_100g": "Prot?ines (g/100g)",
    "salt_100g": "Sel (g/100g)",
    "cholesterol_100g": "Cholest?rol (mg/100g)",
    "vitamin-a_100g": "Vitamine A (?g/100g)",
    "vitamin-c_100g": "Vitamine C (mg/100g)",
    "vitamin-d_100g": "Vitamine D (?g/100g)",
    "calcium_100g": "Calcium (mg/100g)",
    "magnesium_100g": "Magn?sium (mg/100g)",
    "zinc_100g": "Zinc (mg/100g)",
    "manganese_100g": "Mangan?se (mg/100g)",
    "caffeine_100g": "Caf?ine (mg/100g)",
    "alcohol_100g": "Alcool (g/100g)",
    "nutriscore_grade": "Nutri-Score",
    "product_name": "Nom du Produit"
}

df_nutriscore.rename(columns=renommage_colonnes, inplace=True)

df_nutriscore.reset_index(drop=True, inplace=True)

# Sauvegarder
df_nutriscore.to_csv("../datas/FoodProduct_modifier.csv", index=False, encoding='utf-8')

print("? Donn?es nettoy?es et enrichies sauvegard?es.")
print(df_nutriscore.head())



In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# On part de df_nutriscore d?j? nettoy?

# 1. S?lectionner colonnes num?riques (exclure Nutri-Score)
colonnes_numeriques = df_nutriscore.select_dtypes(include=['number']).columns

# 2. Calcul de la matrice de corr?lation
matrice_corr = df_nutriscore[colonnes_numeriques].corr()

print(matrice_corr)

# 3. (Optionnel) Visualisation avec seaborn
plt.figure(figsize=(12, 10))
sns.heatmap(matrice_corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Matrice de corr?lation des variables nutritionnelles")
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns


# 1. Pr?parer X et y
X = df_nutriscore.drop(columns=['Nutri-Score'])
X = X.select_dtypes(include=['number'])  # garder que les colonnes num?riques

y = df_nutriscore['Nutri-Score']

# 2. Encoder les labels Nutri-Score (A,B,C,D,E) en nombres 0-4
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)

# 3. S?parer en train/test (stratification pour conserver proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

# 4. Cr?er le mod?le Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Pr?dictions sur test
y_pred = model.predict(X_test)

# 6. ?valuer le mod?le
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy sur test : {accuracy:.2f}\n")

print("Rapport de classification :")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# 7. Afficher l'importance des variables
importances = model.feature_importances_
features = X.columns
feat_importances = pd.Series(importances, index=features).sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=feat_importances.values, y=feat_importances.index)
plt.title("Importance des variables dans la pr?diction du Nutri-Score")
plt.xlabel("Importance")
plt.ylabel("Composants nutritionnels")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le fichier propre
df = pd.read_csv("../datas/FoodProduct_modifier.csv", encoding='utf-8')

# Nettoyer la colonne Nutri-Score (enlever espaces, mettre en majuscules)
df['Nutri-Score'] = df['Nutri-Score'].str.strip().str.upper()

# Convertir la colonne sucre en num?rique, forcer les erreurs en NaN
df['Sucres (g/100g)'] = pd.to_numeric(df['Sucres (g/100g)'], errors='coerce')

# Supprimer les lignes o? Nutri-Score n'est pas dans ['A','B','C','D','E']
df = df[df['Nutri-Score'].isin(['A', 'B', 'C', 'D', 'E'])]

# Calculer la moyenne de sucre par Nutri-Score (en ignorant les NaN)
moyenne_sucre_par_score = df.groupby('Nutri-Score')['Sucres (g/100g)'].mean()

# R?ordonner selon l'ordre Nutri-Score (A ? E)
ordre = ['A', 'B', 'C', 'D', 'E']
moyenne_sucre_par_score = moyenne_sucre_par_score.reindex(ordre)

# Afficher la moyenne en console
print("Moyenne des sucres (g/100g) par Nutri-Score :")
print(moyenne_sucre_par_score)

# Tracer un graphique en barres
moyenne_sucre_par_score.plot(kind='bar', color='skyblue')
plt.title("Moyenne des sucres (g/100g) par Nutri-Score")
plt.xlabel("Nutri-Score")
plt.ylabel("Moyenne des sucres (g/100g)")
plt.ylim(0)  # commence ? z?ro pour une meilleure lisibilit?
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le fichier propre
df = pd.read_csv("../datas/FoodProduct_modifier.csv", encoding='utf-8')

# Nettoyer la colonne Nutri-Score (enlever espaces, mettre en majuscules)
df['Nutri-Score'] = df['Nutri-Score'].str.strip().str.upper()

# Convertir la colonne mati?re grasse satur?e en num?rique, forcer les erreurs en NaN
# Change ici le nom de la colonne si c?est juste "Mati?res grasses" ou autre
df['Graisses satur?es (g/100g)'] = pd.to_numeric(df['Graisses satur?es (g/100g)'], errors='coerce')

# Supprimer les lignes o? Nutri-Score n'est pas dans ['A','B','C','D','E']
df = df[df['Nutri-Score'].isin(['A', 'B', 'C', 'D', 'E'])]

# Calculer la moyenne de graisses satur?es par Nutri-Score (en ignorant les NaN)
moyenne_graisses_par_score = df.groupby('Nutri-Score')['Graisses satur?es (g/100g)'].mean()

# R?ordonner selon l'ordre Nutri-Score (A ? E)
ordre = ['A', 'B', 'C', 'D', 'E']
moyenne_graisses_par_score = moyenne_graisses_par_score.reindex(ordre)

# Afficher la moyenne en console
print("Moyenne des graisses satur?es (g/100g) par Nutri-Score :")
print(moyenne_graisses_par_score)

# Tracer un graphique en barres
moyenne_graisses_par_score.plot(kind='bar', color='salmon')
plt.title("Moyenne des graisses satur?es (g/100g) par Nutri-Score")
plt.xlabel("Nutri-Score")
plt.ylabel("Moyenne des graisses satur?es (g/100g)")
plt.ylim(0)  # commence ? z?ro pour une meilleure lisibilit?
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le fichier propre
df = pd.read_csv("../datas/FoodProduct_modifier.csv", encoding='utf-8')

# Nettoyer la colonne Nutri-Score (enlever espaces, mettre en majuscules)
df['Nutri-Score'] = df['Nutri-Score'].str.strip().str.upper()

# Convertir la colonne ?nergie en num?rique (forcer les erreurs en NaN si n?cessaire)
df['?nergie (kJ/100g)'] = pd.to_numeric(df['?nergie (kJ/100g)'], errors='coerce')

# Supprimer les lignes o? Nutri-Score n'est pas dans ['A','B','C','D','E']
df = df[df['Nutri-Score'].isin(['A', 'B', 'C', 'D', 'E'])]

# Calculer la moyenne d'?nergie par Nutri-Score (en ignorant les NaN)
moyenne_energie_par_score = df.groupby('Nutri-Score')['?nergie (kJ/100g)'].mean()

# R?ordonner selon l'ordre Nutri-Score (A ? E)
ordre = ['A', 'B', 'C', 'D', 'E']
moyenne_energie_par_score = moyenne_energie_par_score.reindex(ordre)

# Afficher la moyenne en console
print("Moyenne d'?nergie (kJ/100g) par Nutri-Score :")
print(moyenne_energie_par_score)

# Tracer un graphique en barres
moyenne_energie_par_score.plot(kind='bar', color='orange')
plt.title("Moyenne d'?nergie (kJ/100g) par Nutri-Score")
plt.xlabel("Nutri-Score")
plt.ylabel("?nergie moyenne (kJ/100g)")
plt.ylim(0)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le fichier propre
df = pd.read_csv("../datas/FoodProduct_modifier.csv", encoding='utf-8')

# Nettoyer la colonne Nutri-Score
df['Nutri-Score'] = df['Nutri-Score'].str.strip().str.upper()

# Convertir la colonne ?nergie kcal en num?rique
df['?nergie (kcal/100g)'] = pd.to_numeric(df['?nergie (kcal/100g)'], errors='coerce')

# Supprimer les lignes o? Nutri-Score n?est pas dans A-E
df = df[df['Nutri-Score'].isin(['A', 'B', 'C', 'D', 'E'])]

# Calcul de la moyenne ?nergie (kcal) par Nutri-Score
moyenne_energie_kcal = df.groupby('Nutri-Score')['?nergie (kcal/100g)'].mean()

# R?ordonner de A ? E
ordre = ['A', 'B', 'C', 'D', 'E']
moyenne_energie_kcal = moyenne_energie_kcal.reindex(ordre)

# Affichage
print("Moyenne d'?nergie (kcal/100g) par Nutri-Score :")
print(moyenne_energie_kcal)

# Graphe
moyenne_energie_kcal.plot(kind='bar', color='lightcoral')
plt.title("Moyenne d'?nergie (kcal/100g) par Nutri-Score")
plt.xlabel("Nutri-Score")
plt.ylabel("?nergie moyenne (kcal/100g)")
plt.ylim(0)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le fichier
df = pd.read_csv("../datas/FoodProduct_modifier.csv", encoding='utf-8')

# Nettoyer le Nutri-Score
df['Nutri-Score'] = df['Nutri-Score'].str.strip().str.upper()

# Convertir la colonne Vitamine C en num?rique
df['vitamin-c_100g'] = pd.to_numeric(df['Vitamine C (mg/100g)'], errors='coerce')

# Supprimer les lignes o? le Nutri-Score n?est pas A-E
df = df[df['Nutri-Score'].isin(['A', 'B', 'C', 'D', 'E'])]

# Calcul de la moyenne par Nutri-Score
moyenne_vitc = df.groupby('Nutri-Score')['vitamin-c_100g'].mean()

# R?ordonner A ? E
ordre = ['A', 'B', 'C', 'D', 'E']
moyenne_vitc = moyenne_vitc.reindex(ordre)

# Afficher dans la console
print("Moyenne de Vitamine C (mg/100g) par Nutri-Score :")
print(moyenne_vitc)

# Tracer un graphique
moyenne_vitc.plot(kind='bar', color='mediumseagreen')
plt.title("Moyenne de Vitamine C (mg/100g) par Nutri-Score")
plt.xlabel("Nutri-Score")
plt.ylabel("Vitamine C (mg/100g)")
plt.ylim(0)
plt.show()


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import History

# One-hot encoding des labels
y_cat = to_categorical(y_enc)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, stratify=y_cat, random_state=42)

# Mod?le
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(5, activation='softmax'))  # 5 classes Nutri-Score

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Entra?nement avec historique
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Affichage des courbes d'accuracy
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('?volution de la pr?cision au fil des epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()
